In [21]:
print("https://datadriven.io/problems/the_day_7_retention_cohort")

https://datadriven.io/problems/the_day_7_retention_cohort


In [2]:
# =========================
# Imports
# =========================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================
# Create Spark Session
# =========================
spark = (
    SparkSession.builder
    .appName("TopModelsByFramework")
    .getOrCreate()
)

# Optional display setting
spark.conf.set("spark.sql.shuffle.partitions", "4")

# =========================
# Create sample data
# =========================
data = [
    (1,  "FraudModel",    "v1", 0.91, "active",   "2026-01-01", "TensorFlow"),
    (2,  "FraudModel",    "v2", 0.93, "active",   "2026-02-01", "TensorFlow"),
    (3,  "FraudModel",    "v3", 0.88, "inactive", "2026-03-01", "PyTorch"),

    (4,  "PricingModel",  "v1", 0.86, "active",   "2026-01-05", "XGBoost"),
    (5,  "PricingModel",  "v2", 0.89, "active",   "2026-02-05", "LightGBM"),
    (6,  "PricingModel",  "v3", 0.91, "active",   "2026-03-05", "LightGBM"),
    (7,  "PricingModel",  "v4", 0.84, "inactive", "2026-04-05", "XGBoost"),

    (8,  "SearchRanker",  "v1", 0.95, "active",   "2026-01-10", "PyTorch"),
    (9,  "SearchRanker",  "v2", 0.96, "active",   "2026-02-10", "PyTorch"),

    (10, "ForecastModel", "v1", 0.90, "active",   "2026-01-15", "XGBoost"),
    (11, "ForecastModel", "v2", 0.92, "active",   "2026-02-15", "XGBoost"),

    (12, "RecoModel",     "v1", 0.94, "active",   "2026-01-20", "TensorFlow"),
    (13, "RecoModel",     "v2", 0.94, "active",   "2026-02-20", "TensorFlow"),

    (14, "VisionModel",   "v1", 0.94, "active",   "2026-01-25", "PyTorch"),
    (15, "VisionModel",   "v2", 0.94, "active",   "2026-02-25", "PyTorch")
]

columns = [
    "model_id",
    "mdl_name",
    "version",
    "accuracy",
    "status",
    "train_at",
    "framework"
]

ml_models_df = spark.createDataFrame(data, columns)

# Create temp view for Spark SQL
ml_models_df.createOrReplaceTempView("ml_models")

# Check input
spark.sql("SELECT * FROM ml_models").show(truncate=False)

26/06/20 05:16:41 WARN Utils: Your hostname, Prems-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.39 instead (on interface en0)
26/06/20 05:16:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 05:16:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+-------------+-------+--------+--------+----------+----------+
|model_id|mdl_name     |version|accuracy|status  |train_at  |framework |
+--------+-------------+-------+--------+--------+----------+----------+
|1       |FraudModel   |v1     |0.91    |active  |2026-01-01|TensorFlow|
|2       |FraudModel   |v2     |0.93    |active  |2026-02-01|TensorFlow|
|3       |FraudModel   |v3     |0.88    |inactive|2026-03-01|PyTorch   |
|4       |PricingModel |v1     |0.86    |active  |2026-01-05|XGBoost   |
|5       |PricingModel |v2     |0.89    |active  |2026-02-05|LightGBM  |
|6       |PricingModel |v3     |0.91    |active  |2026-03-05|LightGBM  |
|7       |PricingModel |v4     |0.84    |inactive|2026-04-05|XGBoost   |
|8       |SearchRanker |v1     |0.95    |active  |2026-01-10|PyTorch   |
|9       |SearchRanker |v2     |0.96    |active  |2026-02-10|PyTorch   |
|10      |ForecastModel|v1     |0.9     |active  |2026-01-15|XGBoost   |
|11      |ForecastModel|v2     |0.92    |active  |2

In [3]:
result = spark.sql("""
WITH framework_stats AS (
    SELECT
        mdl_name,
        framework,
        COUNT(*) AS version_count,
        AVG(accuracy) AS avg_accuracy
    FROM ml_models
    GROUP BY
        mdl_name,
        framework
),

chosen_framework AS (
    SELECT
        mdl_name,
        framework,
        version_count,
        avg_accuracy,
        ROW_NUMBER() OVER (
            PARTITION BY mdl_name
            ORDER BY
                version_count DESC,
                avg_accuracy DESC,
                framework ASC
        ) AS rn
    FROM framework_stats
),

ranked_models AS (
    SELECT
        mdl_name,
        framework,
        version_count,
        avg_accuracy,
        DENSE_RANK() OVER (
            ORDER BY avg_accuracy DESC
        ) AS accuracy_rank
    FROM chosen_framework
    WHERE rn = 1
)

SELECT
    mdl_name,
    framework,
    version_count,
    ROUND(avg_accuracy, 4) AS avg_accuracy,
    accuracy_rank
FROM ranked_models
WHERE accuracy_rank <= 3
ORDER BY
    accuracy_rank,
    avg_accuracy DESC,
    mdl_name
""")

result.show(truncate=False)

26/06/20 05:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:16:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:16:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:16:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 0

+------------+----------+-------------+------------+-------------+
|mdl_name    |framework |version_count|avg_accuracy|accuracy_rank|
+------------+----------+-------------+------------+-------------+
|SearchRanker|PyTorch   |2            |0.955       |1            |
|RecoModel   |TensorFlow|2            |0.94        |2            |
|VisionModel |PyTorch   |2            |0.94        |2            |
|FraudModel  |TensorFlow|2            |0.92        |3            |
+------------+----------+-------------+------------+-------------+



In [5]:
# The incident response team is prioritizing which services need the most reliability investment.

In [ ]:
# Show each service alongside its alert count and its rank by volume, sorted from most to least.


# alert_events
# alert_id
# INTEGER
# svc_name
# TEXT
# severity
# TEXT
# status
# TEXT
# fired_at
# TEXT
# ack_by
# TEXT
# resolved
# TEXT

In [7]:
# select svc_name, COUNT(alert_id) cnt  from alert_events group by svc_name order by cnt desc

In [8]:

alert_events
alert_id	svc_name	severity	status	fired_at	ack_by	resolved
129	payment-api	high	acknowledged	2026-02-02 01:07:00	bob	2026-02-02 03:11:00
158	user-svc	medium	resolved	2026-03-03 02:14:00	charlie	2026-03-03 04:22:00
187	search-api	low	silenced	2026-04-04 03:21:00	dana	NULL
216	gateway	Critical	firing	2026-05-05 04:28:00	NULL	2026-05-05 06:44:00
245	notif-svc	HIGH	acknowledged	2026-06-06 05:35:00	alice	2026-06-06 07:55:00
274	db-primary	critical	resolved	2026-07-07 06:42:00	bob	NULL
303	cache-01	high	silenced	2026-08-08 07:49:00	charlie	2026-08-08 09:17:00
332	auth-svc	medium	firing	2026-09-09 08:56:00	NULL	2026-09-09 10:28:00
361	payment-api	low	acknowledged	2026-10-10 09:03:00	eve	NULL
390	user-svc	Critical	resolved	2026-11-11 10:10:00	alice	2026-11-11 12:50:00


SyntaxError: invalid syntax (3848790036.py, line 2)

In [11]:


data = [
    (129, "payment-api", "high",     "acknowledged", "2026-02-02 01:07:00", "bob",     "2026-02-02 03:11:00"),
    (158, "user-svc",    "medium",   "resolved",     "2026-03-03 02:14:00", "charlie", "2026-03-03 04:22:00"),
    (187, "search-api",  "low",      "silenced",     "2026-04-04 03:21:00", "dana",    None),
    (216, "gateway",     "Critical", "firing",       "2026-05-05 04:28:00", None,      "2026-05-05 06:44:00"),
    (245, "notif-svc",   "HIGH",     "acknowledged", "2026-06-06 05:35:00", "alice",   "2026-06-06 07:55:00"),
    (274, "db-primary",  "critical", "resolved",     "2026-07-07 06:42:00", "bob",     None),
    (303, "cache-01",    "high",     "silenced",     "2026-08-08 07:49:00", "charlie", "2026-08-08 09:17:00"),
    (332, "auth-svc",    "medium",   "firing",       "2026-09-09 08:56:00", None,      "2026-09-09 10:28:00"),
    (361, "payment-api", "low",      "acknowledged", "2026-10-10 09:03:00", "eve",     None),
    (390, "user-svc",    "Critical", "resolved",     "2026-11-11 10:10:00", "alice",   "2026-11-11 12:50:00")
]

columns = [
    "alert_id",
    "svc_name",
    "severity",
    "status",
    "fired_at",
    "ack_by",
    "resolved"
]

df = spark.createDataFrame(data, columns)

df = (
    df
    .withColumn("fired_at", F.to_timestamp("fired_at"))
    .withColumn("resolved", F.to_timestamp("resolved"))
)

df.createOrReplaceTempView("alert_events")

spark.sql("SELECT * FROM alert_events").show(truncate=False)

+--------+-----------+--------+------------+-------------------+-------+-------------------+
|alert_id|svc_name   |severity|status      |fired_at           |ack_by |resolved           |
+--------+-----------+--------+------------+-------------------+-------+-------------------+
|129     |payment-api|high    |acknowledged|2026-02-02 01:07:00|bob    |2026-02-02 03:11:00|
|158     |user-svc   |medium  |resolved    |2026-03-03 02:14:00|charlie|2026-03-03 04:22:00|
|187     |search-api |low     |silenced    |2026-04-04 03:21:00|dana   |NULL               |
|216     |gateway    |Critical|firing      |2026-05-05 04:28:00|NULL   |2026-05-05 06:44:00|
|245     |notif-svc  |HIGH    |acknowledged|2026-06-06 05:35:00|alice  |2026-06-06 07:55:00|
|274     |db-primary |critical|resolved    |2026-07-07 06:42:00|bob    |NULL               |
|303     |cache-01   |high    |silenced    |2026-08-08 07:49:00|charlie|2026-08-08 09:17:00|
|332     |auth-svc   |medium  |firing      |2026-09-09 08:56:00|NULL  

In [14]:
spark.sql("select svc_name, severity,COUNT(alert_id) cnt  from alert_events group by severity,svc_name order by cnt desc").show(truncate=False)

+-----------+--------+---+
|svc_name   |severity|cnt|
+-----------+--------+---+
|payment-api|high    |1  |
|user-svc   |medium  |1  |
|search-api |low     |1  |
|gateway    |Critical|1  |
|notif-svc  |HIGH    |1  |
|db-primary |critical|1  |
|cache-01   |high    |1  |
|auth-svc   |medium  |1  |
|payment-api|low     |1  |
|user-svc   |Critical|1  |
+-----------+--------+---+



In [16]:
def sql_run(sql_df):
    spark.sql(sql_df).show()

In [17]:
sql_query="""

"""

In [18]:
ql_run(sql_query)

NameError: name 'ql_run' is not defined

In [19]:
result = spark.sql("""
WITH service_alert_counts AS (
    SELECT
        svc_name,
        COUNT(*) AS alert_count
    FROM alert_events
    GROUP BY svc_name
),

ranked_services AS (
    SELECT
        svc_name,
        alert_count,
        DENSE_RANK() OVER (
            ORDER BY alert_count DESC
        ) AS volume_rank
    FROM service_alert_counts
)

SELECT
    svc_name,
    alert_count,
    volume_rank
FROM ranked_services
ORDER BY
    alert_count DESC,
    svc_name
""")

result.show(truncate=False)

+-----------+-----------+-----------+
|svc_name   |alert_count|volume_rank|
+-----------+-----------+-----------+
|payment-api|2          |1          |
|user-svc   |2          |1          |
|auth-svc   |1          |2          |
|cache-01   |1          |2          |
|db-primary |1          |2          |
|gateway    |1          |2          |
|notif-svc  |1          |2          |
|search-api |1          |2          |
+-----------+-----------+-----------+



26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 05:27:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/20 0

In [20]:
# The Day-7 Retention Cohort
# For each weekly signup cohort, compute what percentage of users came back and performed an action on day 7 or later. Show the cohort week, total signups, retained users, and retention rate.
# users
# user_id
# INTEGER
# signup_date
# TEXT
# page_views
# view_id
# INTEGER
# page_url
# TEXT
# user_id
# INTEGER
# referrer
# TEXT
# dur_ms
# INTEGER
# device
# TEXT
# viewed_at
# TEXT

In [24]:

spark = (
    SparkSession.builder
    .appName("Day7RetentionCohort")
    .getOrCreate()
)

# =========================
# Users data
# =========================
users_data = [
    (100, "alice", "alice@example.com", "2025-02-02", "inactive", "25-34"),
    (197, "aaron42", "aaron42@example.com", "2026-03-03", "suspended", "35-44"),
    (294, "amelia", "amelia@example.com", "2024-04-04", "pending_verification", "45-54"),
    (391, "arjun", "arjun@example.com", "2025-05-05", "active", "55-64"),
    (488, "ava99", "ava99@example.com", "2026-06-06", "inactive", "65+"),
    (585, "andrew_k", "andrew_k@example.com", "2024-07-07", "suspended", None),
    (682, "anika", "anika@example.com", "2025-08-08", "pending_verification", "18-24"),
    (779, "aiden", "aiden@example.com", "2026-09-09", "active", "25-34"),
    (876, "aria_b", "aria_b@example.com", "2024-10-10", "inactive", "35-44"),
    (973, "brian", "brian@example.com", "2025-11-11", "suspended", "45-54"),
]

users_columns = [
    "user_id",
    "username",
    "email",
    "signup_date",
    "account_status",
    "age_bucket"
]

users_df = spark.createDataFrame(users_data, users_columns)

users_df = users_df.withColumn(
    "signup_date",
    F.to_date("signup_date")
)

# =========================
# Page views data
# =========================
page_views_data = [
    (5053, "/products", 1555, "amazon.com", 637, "desktop", "2026-12-02 01:09:00"),
    (5106, "/checkout", 1652, "twitter.com", 774, "tablet", "2026-12-03 02:18:00"),
    (5159, "/profile", 1749, "github.com", 911, "Mobile", "2026-12-04 03:27:00"),
    (5212, "/search", 1846, None, 1048, "DESKTOP", "2026-12-05 04:36:00"),
    (5265, "/about", 1943, "reddit.com", 1185, "mobile", "2026-12-06 05:45:00"),
    (5318, "/blog/intro", 1555, None, None, "desktop", "2026-12-07 06:54:00"),
    (5371, "/pricing", 1652, "bing.com", 1459, "tablet", "2026-12-08 07:03:00"),
    (5424, "/docs/api", 1749, "news.ycombinator.com", 1596, "Mobile", "2026-12-09 08:12:00"),
    (5477, "/settings", 1846, None, 1733, "DESKTOP", "2026-12-10 09:21:00"),
    (5530, "/Home", 1943, "bing.com", 1870, "mobile", "2026-12-11 10:30:00"),
    
    # Extra rows matching user_ids from users table so retention can be demonstrated
    (6001, "/home", 100, "google.com", 500, "desktop", "2025-02-10 10:00:00"),   # retained, 8 days later
    (6002, "/search", 197, "google.com", 600, "mobile", "2026-03-05 10:00:00"),  # not retained, 2 days later
    (6003, "/product", 294, "google.com", 700, "desktop", "2024-04-20 10:00:00"),# retained
    (6004, "/cart", 391, "google.com", 800, "mobile", "2025-05-20 10:00:00"),   # retained
    (6005, "/home", 488, "google.com", 500, "desktop", "2026-06-06 10:00:00"),  # not retained, same day
]

page_views_columns = [
    "view_id",
    "page_url",
    "user_id",
    "referrer",
    "dur_ms",
    "device",
    "viewed_at"
]

page_views_df = spark.createDataFrame(page_views_data, page_views_columns)

page_views_df = page_views_df.withColumn(
    "viewed_at",
    F.to_timestamp("viewed_at")
)

# =========================
# Create temp views also
# =========================
users_df.createOrReplaceTempView("users")
page_views_df.createOrReplaceTempView("page_views")

users_df.show(truncate=False)
page_views_df.show(truncate=False)

26/06/20 06:02:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+--------+--------------------+-----------+--------------------+----------+
|user_id|username|email               |signup_date|account_status      |age_bucket|
+-------+--------+--------------------+-----------+--------------------+----------+
|100    |alice   |alice@example.com   |2025-02-02 |inactive            |25-34     |
|197    |aaron42 |aaron42@example.com |2026-03-03 |suspended           |35-44     |
|294    |amelia  |amelia@example.com  |2024-04-04 |pending_verification|45-54     |
|391    |arjun   |arjun@example.com   |2025-05-05 |active              |55-64     |
|488    |ava99   |ava99@example.com   |2026-06-06 |inactive            |65+       |
|585    |andrew_k|andrew_k@example.com|2024-07-07 |suspended           |NULL      |
|682    |anika   |anika@example.com   |2025-08-08 |pending_verification|18-24     |
|779    |aiden   |aiden@example.com   |2026-09-09 |active              |25-34     |
|876    |aria_b  |aria_b@example.com  |2024-10-10 |inactive            |35-4

In [26]:
# =========================
# Step 1: Create signup cohort
# =========================
signup_users_df = (
    users_df
    .select(
        "user_id",
        "signup_date",
        F.concat(
            F.year("signup_date"),
            F.lit("-W"),
            F.lpad(F.weekofyear("signup_date").cast("string"), 2, "0")
        ).alias("cohort_week")
    )
)

# =========================
# Step 2: Join page views and check day-7 retention
# =========================
joined_df = (
    signup_users_df.alias("u")
    .join(
        page_views_df.alias("p"),
        on=(
            (F.col("u.user_id") == F.col("p.user_id")) &
            (F.datediff(F.to_date(F.col("p.viewed_at")), F.col("u.signup_date")) >= 7)
        ),
        how="left"
    )
)

# =========================
# Step 3: User-level retention flag
# =========================
user_retention_df = (
    joined_df
    .groupBy(
        F.col("u.user_id").alias("user_id"),
        F.col("u.cohort_week").alias("cohort_week")
    )
    .agg(
        F.when(F.count("p.view_id") > 0, F.lit(1))
         .otherwise(F.lit(0))
         .alias("is_retained")
    )
)

# =========================
# Step 4: Cohort-level summary
# =========================
result_df = (
    user_retention_df
    .groupBy("cohort_week")
    .agg(
        F.count("*").alias("total_signups"),
        F.sum("is_retained").alias("retained_users")
    )
    .withColumn(
        "retention_pct",
        F.round(100.0 * F.col("retained_users") / F.col("total_signups"), 2)
    )
    .orderBy("cohort_week")
)

result_df.show(truncate=False)

+-----------+-------------+--------------+-------------+
|cohort_week|total_signups|retained_users|retention_pct|
+-----------+-------------+--------------+-------------+
|2024-W14   |1            |1             |100.0        |
|2024-W27   |1            |0             |0.0          |
|2024-W41   |1            |0             |0.0          |
|2025-W05   |1            |1             |100.0        |
|2025-W19   |1            |1             |100.0        |
|2025-W32   |1            |0             |0.0          |
|2025-W46   |1            |0             |0.0          |
|2026-W10   |1            |0             |0.0          |
|2026-W23   |1            |0             |0.0          |
|2026-W37   |1            |0             |0.0          |
+-----------+-------------+--------------+-------------+



In [27]:
result_sql = spark.sql("""
WITH signup_users AS (
    SELECT
        user_id,
        TO_DATE(signup_date) AS signup_date,
        CONCAT(
            YEAR(TO_DATE(signup_date)),
            '-W',
            LPAD(WEEKOFYEAR(TO_DATE(signup_date)), 2, '0')
        ) AS cohort_week
    FROM users
),

user_retention AS (
    SELECT
        su.user_id,
        su.cohort_week,
        CASE
            WHEN COUNT(pv.view_id) > 0 THEN 1
            ELSE 0
        END AS is_retained
    FROM signup_users su
    LEFT JOIN page_views pv
        ON su.user_id = pv.user_id
       AND DATEDIFF(TO_DATE(pv.viewed_at), su.signup_date) >= 7
    GROUP BY
        su.user_id,
        su.cohort_week
),

cohort_summary AS (
    SELECT
        cohort_week,
        COUNT(*) AS total_signups,
        SUM(is_retained) AS retained_users
    FROM user_retention
    GROUP BY cohort_week
)

SELECT
    cohort_week,
    total_signups,
    retained_users,
    ROUND(100.0 * retained_users / total_signups, 2) AS retention_pct
FROM cohort_summary
ORDER BY cohort_week
""")

result_sql.show(truncate=False)

+-----------+-------------+--------------+-------------+
|cohort_week|total_signups|retained_users|retention_pct|
+-----------+-------------+--------------+-------------+
|2024-W14   |1            |1             |100.00       |
|2024-W27   |1            |0             |0.00         |
|2024-W41   |1            |0             |0.00         |
|2025-W05   |1            |1             |100.00       |
|2025-W19   |1            |1             |100.00       |
|2025-W32   |1            |0             |0.00         |
|2025-W46   |1            |0             |0.00         |
|2026-W10   |1            |0             |0.00         |
|2026-W23   |1            |0             |0.00         |
|2026-W37   |1            |0             |0.00         |
+-----------+-------------+--------------+-------------+



In [28]:
# SELECT
#   STRFTIME('%Y-W%W', u.signup_date) AS cohort_week,
#   COUNT(DISTINCT u.user_id) AS total_signups,
#   COUNT(DISTINCT 
#     CASE
#       WHEN JULIANDAY(a.viewed_at) - JULIANDAY(u.signup_date) >= 7 
#       THEN a.user_id
#     END
#   ) AS retained_users,
#   ROUND(
#     CAST(
#       COUNT(DISTINCT 
#         CASE
#           WHEN JULIANDAY(a.viewed_at) - JULIANDAY(u.signup_date) >= 7 
#           THEN a.user_id
#         END
#       ) AS DOUBLE
#     ) / COUNT(DISTINCT u.user_id) * 100,
#     1
#   ) AS retention_pct
# FROM users AS u
# LEFT JOIN page_views AS a
#   ON u.user_id = a.user_id
# WHERE JULIANDAY('now') - JULIANDAY(u.signup_date) >= 7
# GROUP BY cohort_week
# ORDER BY cohort_week;


In [ ]:
# WITH ranked_msgs AS (
#     SELECT
#         topic,
#         offset,
#         LAG(offset) OVER (
#             PARTITION BY topic
#             ORDER BY offset, msg_id
#         ) AS previous_offset,
#         DENSE_RANK() OVER (
#             PARTITION BY topic
#             ORDER BY offset
#         ) AS offset_dense_rank,
#         ROW_NUMBER() OVER (
#             PARTITION BY topic
#             ORDER BY offset, msg_id
#         ) AS offset_row_number
#     FROM stream_msgs
# )

# SELECT
#     topic,
#     offset,
#     previous_offset,
#     offset_dense_rank,
#     offset_row_number
# FROM ranked_msgs
# WHERE previous_offset IS NOT NULL
# ORDER BY
#     topic,
#     offset,
#     offset_row_number;


In [29]:
# > The incident response team is prioritizing which services need the most reliability investment. Show each service alongside its alert count and its rank by volume, sorted from most to least.
# alert_events
# alert_id
# INTEGER
# svc_name
# TEXT
# severity
# TEXT
# status
# TEXT
# fired_at
# TEXT
# ack_by
# TEXT
# resolved
# TEXT

In [30]:


# with srv_events as (
# select 
#  svc_name, COUNT(alert_id) as incident_cnt
#  FROM 
# alert_events
#  GROUP BY svc_name

# )
#  select 
#    svc_name,
#    incident_cnt,
   
#    dense_rank () over ( order by incident_cnt desc ) rank
#    FROM srv_events



In [31]:
# WITH ml_model AS (
#     SELECT
#         mdl_name,
#         framework,
#         COUNT(version) AS version_cnt,
#         ROUND(AVG(accuracy), 3) AS avg_acc
#     FROM ml_models
#     GROUP BY mdl_name, framework
# ),
# latest_model AS (
#     SELECT
#         mdl_name,
#         framework,
#         avg_acc,
#         ROW_NUMBER() OVER (
#             PARTITION BY mdl_name
#             ORDER BY version_cnt DESC, avg_acc DESC
#         ) AS row_num
#     FROM ml_model
# ),
# ranked AS (
#     SELECT
#         mdl_name,
#         framework,
#         avg_acc,
#         RANK() OVER (ORDER BY avg_acc DESC) AS rnk
#     FROM latest_model
#     WHERE row_num = 1
# )
# SELECT mdl_name, framework, avg_acc
# FROM ranked
# WHERE rnk <= 3
# ORDER BY avg_acc DESC;





In [32]:
# WITH session_views AS (
#     SELECT
#         us.session_id,
#         pv.view_id,
#         pv.dur_ms,
#         ROW_NUMBER() OVER (
#             PARTITION BY us.session_id
#             ORDER BY pv.view_id ASC
#         ) AS rn_first,
#         ROW_NUMBER() OVER (
#             PARTITION BY us.session_id
#             ORDER BY pv.view_id DESC
#         ) AS rn_last,
#         COUNT(*) OVER (
#             PARTITION BY us.session_id
#         ) AS view_count
#     FROM user_sessions us
#     JOIN page_views pv ON us.user_id = pv.user_id
# ),
# distances AS (
#     SELECT
#         session_id,
#         MAX(CASE WHEN rn_last  = 1 THEN dur_ms END)
#       - MAX(CASE WHEN rn_first = 1 THEN dur_ms END) AS distance
#     FROM session_views
#     WHERE view_count > 1          -- discard single-view sessions
#     GROUP BY session_id
# )
# SELECT ROUND(AVG(distance), 3) AS avg_distance
# FROM distances;


In [33]:
# WITH valid AS (
#   SELECT
#     mdl_name,
#     accuracy,
#     ROW_NUMBER() OVER (
#       PARTITION BY mdl_name
#       ORDER BY train_at DESC, model_id DESC
#     ) AS rn,
#     COUNT(*) OVER (PARTITION BY mdl_name) AS n_versions
#   FROM ml_models
#   WHERE accuracy IS NOT NULL
#     AND train_at IS NOT NULL
#     AND train_at <> ''
# ),
# agg AS (
#   SELECT
#     mdl_name,
#     n_versions,
#     MAX(CASE WHEN rn = 1 THEN accuracy END) AS latest_accuracy,
#     AVG(CASE WHEN rn > 1 THEN accuracy END) AS avg_previous_accuracy
#   FROM valid
#   GROUP BY mdl_name, n_versions
# )
# SELECT
#   mdl_name AS model_name,
#   ROUND(COALESCE(avg_previous_accuracy, latest_accuracy), 2) AS avg_lifetime_accuracy,
#   ROUND(latest_accuracy, 2) AS latest_accuracy,
#   CASE
#     WHEN n_versions = 1 THEN 0
#     ELSE ROUND(latest_accuracy - avg_previous_accuracy, 2)
#   END AS difference
# FROM agg
# ORDER BY model_name;